# Download and audit

Scope: Download and audit in the WESAD protocol.

Input: Official archive or synthetic generator.

Output and saved artifacts: `qc/dataset_audit.csv`, `alignment.csv`, `questionnaires.json`, `audit_status.json` (under the configured output root; real `.parquet`, synthetic `.csv`).

Dependencies: Environment from `requirements.txt`, `src/` modules; setup dependencies only in notebook 00.

With raw files only, the notebook produces/consults `artifacts/real_raw_audit/` and does not build unsynchronized merges.


In [3]:
from pathlib import Path
import os, sys
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'configs/config.yaml').is_file() and (p / 'src').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter inside the project folder; no machine-specific path is required')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.config import load, setup, smoke_config
from src import workflow as w
SYNTHETIC = os.environ.get('WESAD_SYNTHETIC', '0') == '1'
c = load()
if SYNTHETIC: c = smoke_config(c)

# Reuse saved protocol instead of silently resetting extraction settings.
if not SYNTHETIC:
    import copy, json
    from src.config import fingerprint
    protocol_file = w.root(c) / 'protocol_lock.json'
    if protocol_file.is_file():
        from src.provenance import verify_lock
        saved_protocol = verify_lock(protocol_file)
        frozen = saved_protocol['config']
        if fingerprint(frozen) != saved_protocol['config_sha256']:
            raise ValueError('Saved protocol hash mismatch')
        c = copy.deepcopy(frozen)

setup(c)
print('Data mode:', 'SYNTHETIC EXECUTION ONLY' if SYNTHETIC else 'REAL WESAD')


Data mode: REAL WESAD


In [4]:
from pathlib import Path
from src.wesad_io import discover

if SYNTHETIC:
    from src.synthetic import make_records
    records = make_records(seed=c["splits"]["seed"])
else:
    # Cerca i dati nella cartella raw configurata, incluse le sottocartelle.
    raw_dir = Path(c["dataset"]["raw_dir"])
    if not raw_dir.is_absolute():
        raw_dir = ROOT / raw_dir

    c["dataset"]["raw_dir"] = str(raw_dir.resolve())

    # Impostare True per i file del dataset ufficiale WESAD.
    c["dataset"]["trusted_pickle"] = True

    subject_files = discover(c["dataset"]["raw_dir"])
    #print(f"Trovati {len(subject_files)} soggetti in {raw_dir}")

    records = w.load_records(c)

print(f"Caricati {len(records)} soggetti")

Caricati 15 soggetti
